# Parquet Format Deep Dive

Exploring the Apache Parquet format, compression, and performance optimization.

In [ ]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import numpy as np
from pathlib import Path
import os

## Understanding Parquet Format

In [ ]:
# Create a sample DataFrame
np.random.seed(42)
df = pd.DataFrame({
    'id': range(1000),
    'value': np.random.randint(0, 1000, 1000),
    'category': np.random.choice(['A', 'B', 'C'], 1000),
    'timestamp': pd.date_range('2024-01-01', periods=1000, freq='H'),
    'amount': np.random.uniform(10, 1000, 1000)
})

print(df.head())
print(f"\nShape: {df.shape}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024:.2f} KB")

## Compression Comparison

In [ ]:
# Create output directory
Path('../data/processed').mkdir(parents=True, exist_ok=True)

# Save as CSV for comparison
csv_path = '../data/processed/sample_data.csv'
df.to_csv(csv_path, index=False)
csv_size = os.path.getsize(csv_path) / 1024
print(f"CSV size: {csv_size:.2f} KB")

In [ ]:
# Compare different compression algorithms
compressions = [None, 'snappy', 'gzip', 'brotli']
file_sizes = {}

for compression in compressions:
    filename = f"../data/processed/sample_{compression or 'uncompressed'}.parquet"
    df.to_parquet(filename, compression=compression, index=False)
    size = os.path.getsize(filename) / 1024
    file_sizes[compression or 'uncompressed'] = size
    print(f"Parquet ({compression or 'uncompressed'}): {size:.2f} KB")

print(f"\nCSV baseline: {csv_size:.2f} KB")
print(f"Best compression (snappy): {(1 - file_sizes['snappy']/csv_size)*100:.1f}% reduction")

## Reading Parquet Files

In [ ]:
# Read entire file
parquet_path = '../data/processed/sample_snappy.parquet'
df_loaded = pd.read_parquet(parquet_path)
print("Loaded DataFrame:")
print(df_loaded.head())

In [ ]:
# Read specific columns (efficient!)
df_subset = pd.read_parquet(parquet_path, columns=['id', 'category'])
print("Subset of columns:")
print(df_subset.head())
print(f"Memory: {df_subset.memory_usage(deep=True).sum() / 1024:.2f} KB")

## Parquet Metadata

In [ ]:
# Read parquet file metadata
parquet_file = pq.ParquetFile(parquet_path)

print(f"Number of rows: {parquet_file.metadata.num_rows}")
print(f"Number of columns: {parquet_file.metadata.num_columns}")
print(f"Number of row groups: {parquet_file.metadata.num_row_groups}")
print(f"\nSchema:")
print(parquet_file.schema_arrow)

## Writing Partitioned Parquet

In [ ]:
# Create a larger dataset with year/month
df_large = pd.DataFrame({
    'id': range(10000),
    'value': np.random.randint(0, 1000, 10000),
    'date': pd.date_range('2023-01-01', periods=10000, freq='H'),
})

# Extract year and month
df_large['year'] = df_large['date'].dt.year
df_large['month'] = df_large['date'].dt.month

print(df_large.head())

In [ ]:
# Write with partitioning
output_path = '../data/processed/partitioned'
table = pa.Table.from_pandas(df_large)

pq.write_to_dataset(
    table,
    root_path=output_path,
    partition_cols=['year', 'month'],
    compression='snappy'
)

print("Partitioned Parquet written successfully!")
print(f"Structure: {os.listdir(output_path)}")

In [ ]:
# Read partitioned dataset
dataset = pq.ParquetDataset(output_path)
print(f"Partitions: {dataset.partitions}")
print(f"Schema: {dataset.schema}")

# Read specific partition
df_2023_01 = dataset.read_table(
    filters=[[('year', '==', 2023), ('month', '==', 1)]]
).to_pandas()

print(f"\nRows in 2023-01: {len(df_2023_01)}")

## Performance Benchmarking

In [ ]:
import time

# Compare read performance
csv_path = '../data/processed/sample_data.csv'
parquet_path = '../data/processed/sample_snappy.parquet'

# Read CSV
start = time.time()
for _ in range(10):
    df_csv = pd.read_csv(csv_path)
csv_time = time.time() - start

# Read Parquet
start = time.time()
for _ in range(10):
    df_pq = pd.read_parquet(parquet_path)
parquet_time = time.time() - start

print(f"CSV read time (10x): {csv_time:.3f}s")
print(f"Parquet read time (10x): {parquet_time:.3f}s")
print(f"Speedup: {csv_time/parquet_time:.1f}x")